In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from libpysal.weights import lat2W
from esda import Moran
np.random.seed(12345)

In [ ]:
def extract_year_slice(ds, varname, year):
    da = ds[varname].sel(time=f"{year}-01-01")
    return da.values

In [ ]:
def compute_moran(arr2d):
    H, W = arr2d.shape
    
    # NaN 填零（libpysal 需要）
    flat = np.nan_to_num(arr2d.flatten(), nan=0.0)
    
    w = lat2W(H, W)
    w.transform = "r"
    
    mi = Moran(flat, w, permutations=0)
    return mi.I


In [ ]:
def compute_all_variables(ds, years, output_csv):
    names = ["agri","grassland","forest"] 
    modes = ["basin","region"]

    variables = [f"{m}_{n}" for n in names for m in modes]

    long_results = []

    for varname in variables:
        print(f"\n=== Processing variable: {varname} ===")

        for year in years:
            print(f"Starting {year}")
            arr2d = extract_year_slice(ds, varname, year)
            mi = compute_moran(arr2d)

            long_results.append({
                "year": year,
                "variable": varname,
                "moran": mi
            })

    df_long = pd.DataFrame(long_results)

    df_wide = df_long.pivot(index="year", columns="variable", values="moran")

    df_wide.to_csv(output_csv)
    print(f"Saved CSV to {output_csv}")

    return df_wide


In [ ]:
years = list(range(2010, 2101, 10))
years.insert(0, 2005)

ds = xr.open_dataset(f"../../NC/compare.nc")

df = compute_all_variables(ds, years, output_csv=f"../../CSV/moran/world_moran.csv")
df
